In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path


def find_data_dir():
    target = Path("Data/GEFCom2017QualifyingMatch_3level")
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidate = base / target
        if candidate.exists():
            return candidate
    return Path.cwd()

DATA_DIR = find_data_dir()
HIER_PATH = DATA_DIR / "hierarchy.csv"
SUM_MATRIX_PATH = DATA_DIR / "sum_matrix.csv"
HIER_INFO_PATH = DATA_DIR / "hierarchy_info.json"
NODE_VALUES_PATH = DATA_DIR / "node_values.npy"
NORM_PATH = DATA_DIR / "normalization_params.npy"
NORM_CSV_PATH = DATA_DIR / "node_values_normalized.csv"
DEMAND_OUTPUT_PATH = DATA_DIR / "GEFCom2017QualifyingMatchDemand.csv"

# Bottom level node files
BOTTOM_FILES = {
    "CT": "CT_hourly.csv",
    "ME": "ME_hourly.csv",
    "NEMASSBOST": "NEMASSBOST_hourly.csv",
    "NH": "NH_huorly.csv",
    "RI": "RI_hourly.csv",
    "SEMASS": "SEMASS_hourly.csv",
    "VT": "VT_hourly.csv",
    "WCMASS": "WCMASS_hourly.csv",
}

LOG_OFFSET = 1.0
LOG_SKEW_THRESHOLD = 1.0
LOG_RATIO_THRESHOLD = 10.0
NORM_SKEW_THRESHOLD = 1.0
NORM_KURTOSIS_THRESHOLD = 5.0
TRAIN_RATIO = 0.8  # fit normalization stats on first 80% to avoid test leakage

print(f"Data directory: {DATA_DIR}")
print(f"Hierarchy file: {HIER_PATH}")
print(f"Output file: {DEMAND_OUTPUT_PATH}")


In [ ]:
# Step 1: Read bottom level files and extract DEMAND column
all_data = {}

for node_name, filename in BOTTOM_FILES.items():
    file_path = DATA_DIR / filename
    if not file_path.exists():
        print(f"Warning: {filename} not found")
        continue
    
    df = pd.read_csv(file_path)
    print(f"\nLoading {node_name} from {filename}")
    print(f"Columns: {list(df.columns)}")
    print(f"Shape: {df.shape}")
    print(f"First row: {df.iloc[0].to_dict()}")
    print(f"Hour range: {df['Hour'].min()} to {df['Hour'].max()}")
    
    # Convert Hour from 1-24 to 0-23
    df_copy = df.copy()
    df_copy['Hour'] = df_copy['Hour'] - 1
    print(f"Converted Hour range: {df_copy['Hour'].min()} to {df_copy['Hour'].max()}")
    
    # Ensure DEMAND is numeric (strip thousand separators before coercion)
    if 'DEMAND' in df_copy.columns:
        df_copy['DEMAND'] = pd.to_numeric(df_copy['DEMAND'].astype(str).str.replace(',', ''), errors='coerce')
    
    # Merge Date and Hour to create timestamp
    df_copy['timestamp'] = pd.to_datetime(df_copy['Date'] + ' ' + df_copy['Hour'].astype(str) + ':00:00')
    
    # Extract DEMAND column
    if 'DEMAND' in df_copy.columns:
        demand_data = df_copy[['timestamp', 'DEMAND']].copy()
        demand_data = demand_data.rename(columns={'DEMAND': node_name})
        demand_data = demand_data.set_index('timestamp')
        all_data[node_name] = demand_data
        print(f"Extracted {node_name} demand - Shape: {demand_data.shape}")
        print(f"Time range: {demand_data.index.min()} to {demand_data.index.max()}")
    else:
        print(f"Warning: DEMAND column not found in {filename}")
        print(f"Available columns: {list(df_copy.columns)}")

print(f"\nSuccessfully loaded {len(all_data)} bottom level nodes")


In [ ]:
# Step 2: Merge all bottom level data and create hierarchical structure
import csv
import json
from functools import lru_cache

# Combine all bottom level data
df_combined = pd.concat(all_data.values(), axis=1)
df_combined = df_combined.apply(pd.to_numeric, errors='coerce')  # ensure numeric
df_combined = df_combined.sort_index()
print(f"Combined bottom level data shape: {df_combined.shape}")
print(f"Columns: {list(df_combined.columns)}")
print(df_combined.head())

# Parse hierarchy structure
def as_int_if_possible(value):
    if value is None:
        return None
    text = str(value).strip()
    if text == "":
        return None
    try:
        return int(text)
    except ValueError:
        return text

children = {}
top_order = []
mid_order = []
leaf_order = []

with HIER_PATH.open(newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        top = as_int_if_possible(row.get("Top"))
        mid_raw = as_int_if_possible(row.get("Middle"))
        bottom = as_int_if_possible(row.get("Bottom"))
        mid_text = str(mid_raw) if mid_raw is not None else None
        mid = (mid_text if mid_text.endswith("_mid") else f"{mid_text}_mid") if mid_text is not None else None
        path = [p for p in (top, mid, bottom) if p is not None]
        if not path:
            continue
        if top is not None and top not in top_order:
            top_order.append(top)
        if mid is not None and mid not in mid_order:
            mid_order.append(mid)
        leaf = path[-1]
        if leaf not in leaf_order:
            leaf_order.append(leaf)
        for parent, child in zip(path[:-1], path[1:]):
            children.setdefault(parent, [])
            if child not in children[parent]:
                children[parent].append(child)

print(f"\nHierarchy structure:")
print(f"Top: {top_order}")
print(f"Middle (renamed): {mid_order}")
print(f"Bottom: {leaf_order}")

# Create hierarchical data
bottom_order = leaf_order
node_order = top_order + mid_order + bottom_order

@lru_cache(None)
def get_children(node):
    return children.get(node, [])

# Calculate middle and top level values
df_middle = pd.DataFrame(index=df_combined.index, dtype=float)
df_top = pd.DataFrame(index=df_combined.index, dtype=float)

for mid in mid_order:
    children_list = get_children(mid)
    # Sum only columns that exist in df_combined
    cols_to_sum = [c for c in children_list if c in df_combined.columns]
    if cols_to_sum:
        df_middle[mid] = df_combined[cols_to_sum].sum(axis=1)
    else:
        df_middle[mid] = 0.0
    print(f"Middle node {mid}: sum of {cols_to_sum}")

for top in top_order:
    children_list = get_children(top)
    # Get bottom children (not middle nodes)
    bottom_children = [c for c in children_list if c in df_combined.columns and c not in mid_order]
    if bottom_children:
        df_top[top] = df_combined[bottom_children].sum(axis=1)
    else:
        df_top[top] = 0.0

    # Add contribution from middle nodes that are children
    middle_children = [c for c in children_list if c in mid_order]
    for mid in middle_children:
        df_top[top] = df_top[top].astype(float) + df_middle[mid].astype(float)
    print(f"Top node {top}: sum of bottom {bottom_children} + middle {middle_children}")

# Combine all levels: Top - Middle - Bottom
df_hierarchical = pd.concat([df_top, df_middle, df_combined], axis=1)
df_hierarchical = df_hierarchical[node_order]

print(f"\nHierarchical data shape: {df_hierarchical.shape}")
print(f"Node order: {node_order}")
print(df_hierarchical.head())

# Save to CSV
df_hierarchical.to_csv(DEMAND_OUTPUT_PATH)
print(f"\nSaved hierarchical demand data to {DEMAND_OUTPUT_PATH}")

In [ ]:
# Step 3: Build sum matrix and hierarchy info
bottom_idx = {n: i for i, n in enumerate(bottom_order)}
index_map = {n: i for i, n in enumerate(node_order)}

@lru_cache(None)
def bottoms(node):
    if node in bottom_idx:
        return [node]
    res = []
    for ch in get_children(node):
        for b in bottoms(ch):
            if b not in res:
                res.append(b)
    return res

matrix = [[0] * len(bottom_order) for _ in node_order]
for i, node in enumerate(node_order):
    for b in bottoms(node):
        matrix[i][bottom_idx[b]] = 1

with SUM_MATRIX_PATH.open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerows(matrix)

print(f"Sum matrix shape: ({len(node_order)}, {len(bottom_order)})")
print(f"Sample sum matrix rows:")
for i, row in enumerate(matrix[:3]):
    print(f"  Row {i}: {row}")

# Create hierarchy info
mid_to_bottom_indices = []
for mid in mid_order:
    idxs = []
    for b in bottoms(mid):
        idxs.append(index_map[b])
    mid_to_bottom_indices.append(idxs)

hierarchy_info = {
    "num_total_nodes": len(node_order),
    "num_bottom_nodes": len(bottom_order),
    "bottom_start_idx": len(top_order) + len(mid_order),
    "num_mid_nodes": len(mid_order),
    "num_top_nodes": len(top_order),
    "middle_levels": [list(range(len(top_order), len(top_order) + len(mid_order)))],
    "middle_levels_provenance": {
        "generated_by": "DataProcessing.ipynb",
        "source": "hierarchy.csv",
        "validated_against": ["sum_matrix.csv", "mid_to_bottom_indices"],
    },
    "top_nodes": top_order,
    "mid_nodes": mid_order,
    "bottom_nodes": bottom_order,
    "node_order": node_order,
    "mid_to_bottom_indices": mid_to_bottom_indices,
}

with HIER_INFO_PATH.open("w", encoding="utf-8") as f:
    json.dump(hierarchy_info, f, ensure_ascii=True, indent=2)

print(f"\nHierarchy info:")
print(f"  Total nodes: {hierarchy_info['num_total_nodes']}")
print(f"  Top nodes ({len(top_order)}): {top_order}")
print(f"  Middle nodes ({len(mid_order)}): {mid_order}")
print(f"  Bottom nodes ({len(bottom_order)}): {bottom_order}")
print(f"\nFiles saved:")
print(f"  Sum matrix: {SUM_MATRIX_PATH}")
print(f"  Hierarchy info: {HIER_INFO_PATH}")

In [ ]:
# Step 4: Normalization and prepare final values
from collections import Counter

def log_transform(df, log_offset=1.0):
    return np.log(df + log_offset)

def minmax_normalize(df):
    data = df.values
    min_val = float(data.min())
    max_val = float(data.max())
    denom = max_val - min_val if max_val != min_val else 1.0
    data_norm = (df - min_val) / denom
    params = {"method": "minmax", "min": min_val, "max": max_val}
    return data_norm, params

def zscore_normalize(df):
    data = df.values
    mean_val = float(data.mean())
    std_val = float(data.std())
    denom = std_val if std_val != 0 else 1.0
    data_norm = (df - mean_val) / denom
    params = {"method": "zscore", "mean": mean_val, "std": std_val}
    return data_norm, params

def should_log_transform(series, skew_threshold=1.0, ratio_threshold=10.0):
    values = series.to_numpy(dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return False
    if values.min() < 0:
        return False
    positive = values[values > 0]
    if positive.size == 0:
        return False
    ratio = values.max() / positive.min()
    skew = pd.Series(values).skew()
    if skew is not None and skew > skew_threshold:
        return True
    return ratio > ratio_threshold

def choose_norm_method(series, skew_threshold=1.0, kurtosis_threshold=5.0):
    values = series.to_numpy(dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return "minmax", {"skew": None, "kurtosis": None}
    stats = pd.Series(values)
    skew = float(stats.skew())
    kurtosis = float(stats.kurtosis())
    if abs(skew) <= skew_threshold and abs(kurtosis) <= kurtosis_threshold:
        return "zscore", {"skew": skew, "kurtosis": kurtosis}
    return "minmax", {"skew": skew, "kurtosis": kurtosis}

def normalize_dataframe(df, log_offset=1.0, force_log=None, force_norm=None, train_ratio=TRAIN_RATIO):
    """Fit normalization on the first ``train_ratio`` fraction of rows only,
    then apply to the whole DataFrame.

    This prevents train/test leakage: the test period does not influence the
    log-transform decision, the norm-method decision, nor the fitted
    (mean/std) or (min/max) parameters.
    """
    T = len(df)
    train_T = max(1, int(T * train_ratio))
    df_train = df.iloc[:train_T]

    use_log = force_log if force_log is not None else should_log_transform(
        df_train.stack(),
        skew_threshold=LOG_SKEW_THRESHOLD,
        ratio_threshold=LOG_RATIO_THRESHOLD,
    )

    if use_log:
        df_base = log_transform(df, log_offset=log_offset)
        df_base_train = df_base.iloc[:train_T]
        data_space = "log"
    else:
        df_base = df.copy()
        df_base_train = df_train.copy()
        data_space = "raw"

    norm_method, stats = choose_norm_method(
        df_base_train.stack(),
        skew_threshold=NORM_SKEW_THRESHOLD,
        kurtosis_threshold=NORM_KURTOSIS_THRESHOLD,
    )
    if force_norm is not None:
        norm_method = force_norm

    # Fit on TRAIN, apply to FULL series
    if norm_method == "zscore":
        mean_val = float(df_base_train.values.mean())
        std_val = float(df_base_train.values.std())
        denom = std_val if std_val != 0 else 1.0
        data_norm = (df_base - mean_val) / denom
        norm_params = {"method": "zscore", "mean": mean_val, "std": std_val}
    else:
        min_val = float(df_base_train.values.min())
        max_val = float(df_base_train.values.max())
        denom = max_val - min_val if max_val != min_val else 1.0
        data_norm = (df_base - min_val) / denom
        norm_params = {"method": "minmax", "min": min_val, "max": max_val}

    params = {
        "use_log": bool(use_log),
        "log_offset": float(log_offset) if use_log else None,
        "data_space": data_space,
        "norm_method": norm_method,
        "decision_stats": stats,
        "train_ratio": float(train_ratio),
        "train_T": int(train_T),
        "total_T": int(T),
    }
    params.update(norm_params)

    values = data_norm.to_numpy(dtype=np.float32).reshape(-1, df.shape[1], 1)
    return values, params, use_log, norm_method
# Read hierarchical data and normalize
df_raw = pd.read_csv(DEMAND_OUTPUT_PATH, index_col=0)
df_raw.index = pd.to_datetime(df_raw.index)

# Deduplicate columns that share the same base name (e.g., CT, CT.1, CT.2)
expected_counts = Counter(node_order)
col_buckets = {name: [] for name in expected_counts}
for col in df_raw.columns:
    base = col.split(".")[0]
    if base in expected_counts and len(col_buckets[base]) < expected_counts[base]:
        col_buckets[base].append(col)

missing = [name for name, cols in col_buckets.items() if len(cols) < expected_counts[name]]
if missing:
    raise ValueError(f"Missing expected columns for nodes: {missing}")

selected_cols = []
for name in node_order:
    selected_cols.append(col_buckets[name].pop(0))

df_nodes = df_raw[selected_cols]
df_nodes.columns = node_order  # enforce expected names and order (with duplicates)

print(f"Input data shape: {df_nodes.shape}")
print(f"Columns: {list(df_nodes.columns)}")
print(f"Index range: {df_nodes.index.min()} to {df_nodes.index.max()}")

# Normalize
values, norm_params, use_log, norm_method = normalize_dataframe(df_nodes, log_offset=LOG_OFFSET)

# Save normalized values and parameters
np.save(NODE_VALUES_PATH, values)
np.save(NORM_PATH, norm_params)
df_norm = pd.DataFrame(values.squeeze(-1), index=df_nodes.index, columns=df_nodes.columns)
df_norm.to_csv(NORM_CSV_PATH)
print(f"Saved normalized csv to {NORM_CSV_PATH}")

print(f"\nNormalization completed:")
print(f"  Use log transform: {use_log}")
print(f"  Normalization method: {norm_method}")
print(f"  Normalized values shape: {values.shape}")
print(f"  Normalization parameters: {norm_params}")
print(f"\nFiles saved:")
print(f"  Node values: {NODE_VALUES_PATH}")
print(f"  Normalization params: {NORM_PATH}")